# Monitoring Notebook

Builds a dense monthly monitoring table from Gold data for brand market share.

## Alert Rule
Alert when Brand#32's share in any category falls by at least 5 percentage points compared with the previous comparable month.

`is_comparable_month = true` flags months where all brands in that category have completed their data window. The latest month is excluded because subsequent months may change the share denominator.

The production alert query is in `sql/brand_share_alert.sql`.

In [0]:
%sql
CREATE OR REPLACE TABLE bda_gold.brand_monitoring AS
WITH
brand_categories AS (
    SELECT DISTINCT brand, category FROM bda_gold.brand_sales
),
months AS (
    SELECT DISTINCT month_start FROM bda_gold.brand_sales
),
calendar AS (
    SELECT m.month_start, bc.brand, bc.category
    FROM months m CROSS JOIN brand_categories bc
),
actuals AS (
    SELECT month_start, category, brand, SUM(net_revenue) AS brand_revenue
    FROM bda_gold.brand_sales
    GROUP BY month_start, category, brand
),
filled AS (
    SELECT c.month_start, c.brand, c.category,
        COALESCE(a.brand_revenue, 0) AS brand_revenue
    FROM calendar c
    LEFT JOIN actuals a
        ON c.month_start = a.month_start
        AND c.brand = a.brand AND c.category = a.category
),
with_totals AS (
    SELECT
        f.month_start, f.brand, f.category, f.brand_revenue,
        SUM(f.brand_revenue) OVER (PARTITION BY f.month_start, f.category) AS category_revenue,
        CASE WHEN SUM(f.brand_revenue) OVER (PARTITION BY f.month_start, f.category) = 0
             THEN NULL
             ELSE f.brand_revenue / SUM(f.brand_revenue) OVER (PARTITION BY f.month_start, f.category)
        END AS market_share
    FROM filled f
),
with_lag AS (
    SELECT
        *,
        LAG(market_share) OVER (
            PARTITION BY brand, category ORDER BY month_start
        ) AS previous_market_share
    FROM with_totals
),
max_month AS (
    SELECT MAX(month_start) AS latest_month FROM bda_gold.brand_sales
)
SELECT
    w.month_start,
    w.brand,
    w.category,
    w.brand_revenue,
    w.category_revenue,
    w.market_share,
    w.previous_market_share,
    (w.market_share - w.previous_market_share) * 100 AS share_change_percentage_points,
    CASE WHEN w.month_start < m.latest_month THEN true ELSE false END AS is_complete_month,
    CASE WHEN w.month_start < m.latest_month
              AND w.previous_market_share IS NOT NULL
         THEN true ELSE false END AS is_comparable_month
FROM with_lag w
CROSS JOIN max_month m;

In [0]:
%sql
-- Coverage report: brand-category-months with zero revenue
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN brand_revenue = 0 THEN 1 ELSE 0 END) AS zero_revenue_rows,
    SUM(CASE WHEN market_share IS NULL THEN 1 ELSE 0 END) AS null_share_rows,
    SUM(CASE WHEN is_complete_month THEN 1 ELSE 0 END) AS complete_months,
    SUM(CASE WHEN is_comparable_month THEN 1 ELSE 0 END) AS comparable_months,
    COUNT(DISTINCT month_start) AS distinct_months
FROM bda_gold.brand_monitoring;

-- Latest 20 rows for Brand#32
SELECT
    month_start, category,
    ROUND(brand_revenue, 2) AS brand_revenue,
    ROUND(category_revenue, 2) AS category_revenue,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(previous_market_share * 100, 2) AS prev_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_complete_month, is_comparable_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
ORDER BY month_start DESC, category
LIMIT 20;

In [0]:
%sql
-- Latest comparable month's Brand#32 shares and changes
SELECT
    month_start, category,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_comparable_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
  AND is_comparable_month
  AND month_start = (
      SELECT MAX(month_start)
      FROM bda_gold.brand_monitoring
      WHERE is_comparable_month AND brand = 'Brand#32'
  )
ORDER BY category;

## Alert Demonstration

Real Gold tables are historical and unlikely to trigger the alert.
We create a **synthetic** monitoring result with a 6pp drop for Brand#32 to demonstrate the alert trigger. The real alert query is in `sql/brand_share_alert.sql`.

A second synthetic case with a 3pp change confirms the alert does NOT trigger below the 5pp threshold.

In [0]:
latest_months = spark.sql("""
    SELECT DISTINCT month_start
    FROM bda_gold.brand_monitoring
    WHERE is_comparable_month
    ORDER BY month_start DESC
    LIMIT 2
""").collect()

if len(latest_months) >= 2:
    latest_month = latest_months[0][0]
    prev_month = latest_months[1][0]

    # Synthetic data with 6pp drop for Brand#32
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW demo_brand_monitoring AS
        SELECT
            month_start, brand, category,
            brand_revenue, category_revenue,
            CASE
                WHEN brand = 'Brand#32' AND month_start = '{latest_month}'
                THEN market_share - 0.06
                ELSE market_share
            END AS market_share,
            previous_market_share,
            CASE
                WHEN brand = 'Brand#32' AND month_start = '{latest_month}'
                THEN (market_share - 0.06 - COALESCE(previous_market_share, market_share)) * 100
                ELSE (market_share - COALESCE(previous_market_share, market_share)) * 100
            END AS share_change_percentage_points,
            is_complete_month,
            is_comparable_month
        FROM bda_gold.brand_monitoring
        WHERE brand = 'Brand#32'
    """)

    print(f"Demo created: Brand#32 share dropped 6pp in latest comparable month ({latest_month})")
    print(f"Previous comparable month: {prev_month}\n")

    # Show the synthetic data for the latest month
    print("Demo monitoring data (latest comparable month):")
    spark.sql(f"""
        SELECT month_start, category,
               ROUND(market_share * 100, 2) AS share_pct,
               ROUND(share_change_percentage_points, 2) AS change_pp
        FROM demo_brand_monitoring
        WHERE month_start = '{latest_month}'
        ORDER BY category
    """).show(truncate=False)

    # Test 1: 6pp drop should trigger
    triggered = spark.sql("""
        WITH latest AS (
            SELECT MAX(month_start) AS month_start
            FROM demo_brand_monitoring
            WHERE is_comparable_month
        )
        SELECT COUNT(*) AS affected_category_count
        FROM demo_brand_monitoring m
        JOIN latest ON m.month_start = latest.month_start
        WHERE m.brand = 'Brand#32'
          AND m.share_change_percentage_points <= -5
    """).collect()[0][0]
    print(f"Test 1 (6pp drop): affected_category_count = {triggered}")
    assert triggered > 0, "6pp drop should trigger alert"
    print("  PASS: alert triggers for 6pp drop\n")

    # Test 2: 3pp change should NOT trigger
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW demo_brand_monitoring_3pp AS
        SELECT
            month_start, brand, category,
            brand_revenue, category_revenue,
            CASE
                WHEN brand = 'Brand#32' AND month_start = '{latest_month}'
                THEN market_share - 0.03
                ELSE market_share
            END AS market_share,
            previous_market_share,
            CASE
                WHEN brand = 'Brand#32' AND month_start = '{latest_month}'
                THEN (market_share - 0.03 - COALESCE(previous_market_share, market_share)) * 100
                ELSE (market_share - COALESCE(previous_market_share, market_share)) * 100
            END AS share_change_percentage_points,
            is_complete_month,
            is_comparable_month
        FROM bda_gold.brand_monitoring
        WHERE brand = 'Brand#32'
    """)

    not_triggered = spark.sql("""
        WITH latest AS (
            SELECT MAX(month_start) AS month_start
            FROM demo_brand_monitoring_3pp
            WHERE is_comparable_month
        )
        SELECT COUNT(*) AS affected_category_count
        FROM demo_brand_monitoring_3pp m
        JOIN latest ON m.month_start = latest.month_start
        WHERE m.brand = 'Brand#32'
          AND m.share_change_percentage_points <= -5
    """).collect()[0][0]
    print(f"Test 2 (3pp change): affected_category_count = {not_triggered}")
    assert not_triggered == 0, "3pp change should NOT trigger alert"
    print("  PASS: alert does NOT trigger for 3pp change")
else:
    print("Not enough comparable months for demo")